In [3]:
import requests
import duckdb
import os

# 1. Setup paths
output_dir = '../data_processed'
os.makedirs(output_dir, exist_ok=True)
csv_temp_path = os.path.join(output_dir, 'temp_2025_data.csv')
parquet_final_path = os.path.join(output_dir, 'calls_2025_fresh_api.parquet')

# 2. API Parameters
base_url = "https://data.seattle.gov/resource/33kz-ixgy.csv"
params = {
    "$limit": 1000000,
    "$where": "cad_event_original_time_queued >= '2025-01-01T00:00:00' AND cad_event_original_time_queued < '2026-01-01T00:00:00'"
}

print("📡 Downloading 2025 data via Python requests...")
response = requests.get(base_url, params=params, timeout=120)

if response.status_code == 200:
    with open(csv_temp_path, 'wb') as f:
        f.write(response.content)
    print(f"✅ CSV Downloaded successfully ({len(response.content) / 1024 / 1024:.2f} MB)")
else:
    print(f"❌ Failed to download. Status Code: {response.status_code}")
    print(f"Response: {response.text}")

# 3. Convert CSV to Parquet using DuckDB
if os.path.exists(csv_temp_path):
    print("🔄 Converting CSV to Parquet for performance...")
    con = duckdb.connect()
    con.execute(f"COPY (SELECT * FROM read_csv_auto('{csv_temp_path}')) TO '{parquet_final_path}' (FORMAT PARQUET);")
    
    # 4. Final Sanity Check
    stats = con.execute(f"SELECT COUNT(*) as total_rows, COUNT(DISTINCT cad_event_number) as unique_events FROM '{parquet_final_path}'").df()
    con.close()
    
    # Optional: Remove temp CSV to save space
    # os.remove(csv_temp_path)

    print("\n--- Fresh Data Sanity Check ---")
    print(stats)

📡 Downloading 2025 data via Python requests...
✅ CSV Downloaded successfully (278.57 MB)
🔄 Converting CSV to Parquet for performance...

--- Fresh Data Sanity Check ---
   total_rows  unique_events
0      546208         324386


📡 Downloading 2025 data via Python requests...
✅ CSV Downloaded successfully (278.57 MB)
🔄 Converting CSV to Parquet for performance...

--- Fresh Data Sanity Check ---
   total_rows  unique_events
0      546208         324386

In [9]:
import duckdb
import pandas as pd

con = duckdb.connect()
fresh_file = '../data_processed/calls_2025_fresh_api.parquet'

# 1. Get Schema
print("--- 1. SCHEMA (COLUMN NAMES & TYPES) ---")
schema_df = con.execute(f"DESCRIBE SELECT * FROM '{fresh_file}'").df()
print(schema_df[['column_name', 'column_type']])

# 2. Find the most duplicated ID
top_id_df = con.execute(f"""
    SELECT cad_event_number, COUNT(*) as n 
    FROM '{fresh_file}' 
    GROUP BY 1 ORDER BY 2 DESC LIMIT 1
""").df()

target_id = top_id_df['cad_event_number'].iloc[0]
total_dupes = top_id_df['n'].iloc[0]

# 3. Check for Bit-for-Bit Identity vs. Varying Columns
# We compare COUNT(*) to COUNT(DISTINCT all_columns)
audit_query = f"""
SELECT 
    COUNT(*) as total_rows_for_event,
    -- This counts how many rows are 100% identical across every single column
    (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM '{fresh_file}' WHERE cad_event_number = '{target_id}')) as unique_full_rows,
     COUNT(DISTINCT cad_event_arrived_time) as unique_arrival_times
FROM '{fresh_file}'
WHERE cad_event_number = '{target_id}'
"""

df_audit = con.execute(audit_query).df()
con.close()

print(f"\n--- 2. DUPLICATION ANALYSIS FOR CAD: {target_id} ---")
print(df_audit)

# Logical check to explain the result
if df_audit['total_rows_for_event'].iloc[0] == df_audit['unique_full_rows'].iloc[0]:
    print("\nCONCLUSION: These are LOGICAL DUPLICATES. The rows have differences in some columns (likely different units responding).")
else:
    print("\nCONCLUSION: These are GHOST DUPLICATES. Entire rows are identical.")

--- 1. SCHEMA (COLUMN NAMES & TYPES) ---
                                          column_name column_type
0                                    cad_event_number      BIGINT
1                     cad_event_clearance_description     VARCHAR
2                                           call_type     VARCHAR
3                                            priority      BIGINT
4                                   initial_call_type     VARCHAR
5                                     final_call_type     VARCHAR
6                      cad_event_original_time_queued   TIMESTAMP
7                              cad_event_arrived_time   TIMESTAMP
8                                   dispatch_precinct     VARCHAR
9                                     dispatch_sector     VARCHAR
10                                      dispatch_beat     VARCHAR
11                                 dispatch_longitude     VARCHAR
12                                  dispatch_latitude     VARCHAR
13                            dispa

In [10]:
import duckdb
import pandas as pd

con = duckdb.connect()
fresh_file = '../data_processed/calls_2025_fresh_api.parquet'
target_id = '2025000353113'

# 1. Fetch all 100 rows for this specific incident
df_100 = con.execute(f"SELECT * FROM '{fresh_file}' WHERE cad_event_number = '{target_id}'").df()

# 2. Identify which columns actually have varying data
varying_cols = [col for col in df_100.columns if df_100[col].nunique() > 1]

print(f"--- Inspection of CAD: {target_id} ---")
print(f"Total Rows: {len(df_100)}")
print(f"Columns that differ across these rows: {varying_cols}\n")

# 3. Print the rows, focusing on the columns that change
# We include cad_event_number and the varying columns for context
display_cols = ['cad_event_number'] + varying_cols
print("--- Data for Varying Columns ---")
print(df_100[display_cols].head(20)) # Printing first 20 to see the pattern

con.close()

--- Inspection of CAD: 2025000353113 ---
Total Rows: 100
Columns that differ across these rows: ['call_sign_dispatch_id', 'call_sign_dispatch_time', 'call_sign_total_service_time_s_', 'call_sign_dispatch_delay_time_s_', 'call_sign_response_time_s_', 'call_sign_at_scene_time', 'call_sign_in_service_time', 'count_of_officers']

--- Data for Varying Columns ---
    cad_event_number call_sign_dispatch_id call_sign_dispatch_time  \
0      2025000353113      20250003531132G3     2025-12-02 13:34:38   
1      2025000353113     20250003531132N91     2025-12-02 13:35:30   
2      2025000353113     20250003531132S31     2025-12-02 13:31:55   
3      2025000353113      20250003531132S1     2025-12-02 13:33:48   
4      2025000353113     20250003531132N10     2025-12-02 13:35:43   
5      2025000353113      20250003531132G2     2025-12-02 13:34:37   
6      2025000353113      20250003531132L1     2025-12-02 13:48:57   
7      2025000353113      20250003531132D2     2025-12-02 13:34:28   
8      20

deduping-
--- Step 1: Physical Deduplication Check ---
Raw Row Count: 546208
Distinct Row Count: 546208
Rows Dropped: 0
✅ Confirmed: Every row is physically unique. Duplication is logical (incident logs).

In [11]:
#dedup data based on schema

import duckdb

con = duckdb.connect()
input_path = '../data_processed/calls_2025_fresh_api.parquet'

# Count raw rows
raw_count = con.execute(f"SELECT COUNT(*) FROM '{input_path}'").fetchone()[0]

# Count distinct rows (physical dedup)
physical_dedup_count = con.execute(f"SELECT COUNT(*) FROM (SELECT DISTINCT * FROM '{input_path}')").fetchone()[0]

print(f"--- Step 1: Physical Deduplication Check ---")
print(f"Raw Row Count: {raw_count}")
print(f"Distinct Row Count: {physical_dedup_count}")
print(f"Rows Dropped: {raw_count - physical_dedup_count}")

if raw_count == physical_dedup_count:
    print("✅ Confirmed: Every row is physically unique. Duplication is logical (incident logs).")
else:
    print("⚠️ Notice: Some exact identical rows were found and removed.")

--- Step 1: Physical Deduplication Check ---
Raw Row Count: 546208
Distinct Row Count: 546208
Rows Dropped: 0
✅ Confirmed: Every row is physically unique. Duplication is logical (incident logs).



--- Step 2: Logical Deduplication ---
Final Incident Count (Rows after Logic): 324386
Total Log Records Collapsed: 221822

In [12]:
output_path = '../data_processed/calls_2025_deduplicated_final.parquet'

# Logical Deduplication using the Window Function
logical_dedup_sql = f"""
COPY (
    WITH RankedEvents AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY cad_event_number 
                ORDER BY 
                    (final_call_type IS NOT NULL) DESC,
                    call_sign_in_service_time DESC,
                    cad_event_arrived_time DESC
            ) as rank
        FROM '{input_path}'
    )
    SELECT * EXCLUDE (rank)
    FROM RankedEvents
    WHERE rank = 1
) TO '{output_path}' (FORMAT PARQUET);
"""

print(f"\n--- Step 2: Logical Deduplication ---")
con.execute(logical_dedup_sql)

# Final Count
final_count = con.execute(f"SELECT COUNT(*) FROM '{output_path}'").fetchone()[0]
con.close()

print(f"Final Incident Count (Rows after Logic): {final_count}")
print(f"Total Log Records Collapsed: {physical_dedup_count - final_count}")


--- Step 2: Logical Deduplication ---
Final Incident Count (Rows after Logic): 324386
Total Log Records Collapsed: 221822


In [13]:
import duckdb

con = duckdb.connect()
fresh_file = '../data_processed/calls_2025_fresh_api.parquet'
deduped_file = '../data_processed/calls_2025_deduplicated_final.parquet'

# 1. Count unique IDs in the RAW file
raw_unique_count = con.execute(f"SELECT COUNT(DISTINCT cad_event_number) FROM '{fresh_file}'").fetchone()[0]

# 2. Count total rows in the DEDUPED file
final_row_count = con.execute(f"SELECT COUNT(*) FROM '{deduped_file}'").fetchone()[0]

con.close()

print(f"--- Final Validation Check ---")
print(f"Unique Incident IDs in Raw Data: {raw_unique_count}")
print(f"Total Rows in Deduplicated File: {final_row_count}")

if raw_unique_count == final_row_count:
    print("\n✅ MATCH: The deduplication logic successfully preserved exactly one row for every unique incident.")
else:
    print("\n❌ MISMATCH: The numbers do not match. We need to investigate if some IDs were dropped or multiplied.")

--- Final Validation Check ---
Unique Incident IDs in Raw Data: 324386
Total Rows in Deduplicated File: 324386

✅ MATCH: The deduplication logic successfully preserved exactly one row for every unique incident.
